<a href="https://colab.research.google.com/github/Saibabu7770/bitcal-tts/blob/main/colab_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BitCal-TTS: Bit-Calibrated Test-Time Scaling

**Full experiment on Google Colab (T4 GPU, 15 GB VRAM)**

Supports three model sizes:
- **3B** (~2.5 GB VRAM, ~90 min) — quick validation (100 items)
- **7B** (~4.5 GB VRAM, ~3 hr) — recommended for paper (100 items)
- **14B** (~9 GB VRAM, ~5 hr) — strongest paper results (100 items)

> **Safe to interrupt at any time** — results are written to disk after every single row.

**Steps:** Check GPU → Install → Clone → Smoke test → Run experiment → Analyze → Download

In [1]:
# Cell 1 — Verify GPU and pick model
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu_name}')
    print(f'VRAM: {vram_gb:.1f} GB')
    if vram_gb >= 12:
        print('Recommendation: 7B (fast) or 14B (best accuracy)')
    else:
        print('Recommendation: 3B (low VRAM)')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU')

CUDA available: True
GPU : Tesla T4
VRAM: 15.6 GB
Recommendation: 7B (fast) or 14B (best accuracy)


In [2]:
# Cell 2 — Install dependencies
!pip install -q -U bitsandbytes>=0.46.1
!pip install -q -U transformers>=4.38.0 accelerate>=0.27.0 datasets>=2.18.0 matplotlib>=3.7.0 pyyaml scipy
print('Dependencies installed.')

Dependencies installed.


In [3]:
# Cell 3 — Clone / update repo
import os, subprocess

if not os.path.exists('/content/bitcal-tts'):
    !git clone https://github.com/Saibabu7770/bitcal-tts.git /content/bitcal-tts
else:
    !git -C /content/bitcal-tts pull origin main

os.chdir('/content/bitcal-tts')
!pip install -e . -q
print(f'Repo ready. CWD: {os.getcwd()}')

Cloning into '/content/bitcal-tts'...
remote: Enumerating objects: 224, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 224 (delta 89), reused 198 (delta 63), pack-reused 0 (from 0)
Receiving objects: 100% (224/224), 422.66 KiB | 10.57 MiB/s, done.
Resolving deltas: 100% (89/89), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for bitcal-tts (pyproject.toml) ... done
Repo ready. CWD: /content/bitcal-tts


In [4]:
# Cell 4 — Verify setup
import os
os.chdir('/content/bitcal-tts')
!python -m bitcal_tts doctor

BitCal-TTS 0.1.0

  torch: 2.10.0+cu128 (cuda available: True)
  transformers: 5.5.0
  pyyaml: 6.0.3


In [ ]:
# Cell 5 — Smoke test (1 item, fixed only, confirms model loads)
import os
os.chdir('/content/bitcal-tts')

# Choose model for smoke test (always use 3B for speed)
!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-3B-Instruct \
    --n-samples 1 \
    --budget 64 \
    --methods fixed \
    --source hf \
    --step-size 8 \
    --output-dir results/raw
print('Smoke test done.')

Loading dataset (source=hf, n=1)...
README.md: 7.93kB [00:00, 20.4MB/s]
main/train-00000-of-00001.parquet: 100% 2.31M/2.31M [00:01<00:00, 1.88MB/s]
main/test-00000-of-00001.parquet: 100% 419k/419k [00:00<00:00, 992kB/s] 
Generating train split: 100% 7473/7473 [00:00<00:00, 203641.12 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 109253.85 examples/s]
  Loaded 1 items.
Loading model: Qwen/Qwen2.5-3B-Instruct  (4-bit=True) ...
config.json: 100% 661/661 [00:00<00:00, 4.15MB/s]
tokenizer_config.json: 7.30kB [00:00, 24.8MB/s]
vocab.json: 2.78MB [00:00, 116MB/s]
merges.txt: 1.67MB [00:00, 99.1MB/s]
tokenizer.json: 7.03MB [00:00, 119MB/s]
model.safetensors.index.json: 35.6kB [00:00, 93.0MB/s]

## Cell 6A / 6B / 6C — Choose your model and run

Run **only one** of the three cells below depending on how much time you have.
All cells run **100 items** (900 rows total = 100 items × 3 methods × 3 budgets).
**Safe to interrupt at any time** — every row is saved to disk immediately.

| Cell | Model | VRAM | Time (100 items) | Best for |
|---|---|---|---|---|
| 6A | 3B | 2.5 GB | ~90 min | Quick re-run / validation |
| **6B** | **7B** | **4.5 GB** | **~3 hr** | **Recommended for paper** |
| 6C | 14B | 9 GB | ~5 hr | Strongest paper results |

In [ ]:
# Cell 6A — Qwen2.5-3B-Instruct 4-bit (~90 min, 100 items)
# Safe to interrupt — each row is saved to disk immediately.
import os
os.chdir('/content/bitcal-tts')

!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-3B-Instruct \
    --n-samples 100 \
    --budget 256 512 1024 \
    --methods fixed adaptive bitcal_tts \
    --source hf \
    --step-size 16 \
    --min-tokens-before-halt 128 \
    --bit-width 4 \
    --output-dir results/raw
print('3B experiment complete.')

In [ ]:
# Cell 6B — Qwen2.5-7B-Instruct 4-bit (~3 hr, 100 items) RECOMMENDED FOR PAPER
# Safe to interrupt — each row is saved to disk immediately.
# You will see: [1/900] ... [2/900] ... — stop any time and download results.
import os
os.chdir('/content/bitcal-tts')

!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-7B-Instruct \
    --n-samples 100 \
    --budget 256 512 1024 \
    --methods fixed adaptive bitcal_tts \
    --source hf \
    --step-size 16 \
    --min-tokens-before-halt 128 \
    --bit-width 4 \
    --output-dir results/raw
print('7B experiment complete.')

In [ ]:
# Cell 6C — Qwen2.5-14B-Instruct 4-bit (~5 hr, 100 items) BEST ACCURACY
# Safe to interrupt — each row is saved to disk immediately.
import os
os.chdir('/content/bitcal-tts')

!python scripts/run_experiment.py \
    --model Qwen/Qwen2.5-14B-Instruct \
    --n-samples 100 \
    --budget 512 1024 \
    --methods fixed adaptive bitcal_tts \
    --source hf \
    --step-size 16 \
    --min-tokens-before-halt 128 \
    --bit-width 4 \
    --output-dir results/raw
print('14B experiment complete.')

In [ ]:
# Cell 7 — Analyze results and generate plots
import os
os.chdir('/content/bitcal-tts')
!python scripts/analyze_results.py \
    --results-dir results/raw \
    --out-dir results/processed

In [ ]:
# Cell 8 — Display plots inline
from IPython.display import Image, display
import os

os.chdir('/content/bitcal-tts')
for fname in ['pareto_quality_tokens.png', 'accuracy_by_budget.png']:
    path = f'results/processed/{fname}'
    if os.path.exists(path):
        print(f'\n--- {fname} ---')
        display(Image(path))

In [ ]:
# Cell 9 — Print full summary table
import pandas as pd, os
os.chdir('/content/bitcal-tts')
df = pd.read_csv('results/processed/summary.csv')
# Show only rows from the latest large experiment (n>=50)
df_main = df[df['n'].astype(int) >= 50]
print('=== Main results (n >= 50 items) ===')
print(df_main.to_string(index=False))
print('\n=== Full table ===')
print(df.to_string(index=False))

In [ ]:
# Cell 10 — Download all results as a zip
import shutil, os
from google.colab import files

os.chdir('/content/bitcal-tts')
shutil.make_archive('bitcal_tts_results', 'zip', '.', 'results')
files.download('bitcal_tts_results.zip')
print('Downloaded: bitcal_tts_results.zip')

## Optional: Push results back to GitHub

Create a PAT at https://github.com/settings/tokens (Contents: Read and write)

In [ ]:
# Cell 11 (optional) — Push results to GitHub
import os
os.chdir('/content/bitcal-tts')

GITHUB_TOKEN = ''  # paste your PAT here
GITHUB_EMAIL = 'Saibabu7770@users.noreply.github.com'

if GITHUB_TOKEN:
    !git config user.name 'Sai'
    !git config user.email '{GITHUB_EMAIL}'
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/Saibabu7770/bitcal-tts.git
    !git add results/
    !git commit -m 'Results: Colab T4 GPU experiment'
    !git push origin main
    print('Results pushed to GitHub!')
else:
    print('Skipped: set GITHUB_TOKEN above to push results.')